 # CAPA Dashboard - SQL Queries

Connects to the `medfluss` PostgreSQL database and runs KPI queries for Tableau.

**Input:** `data/capa_medfluss.csv` loaded into PostgreSQL

**Output:** 7 DataFrames: df_status, df_severity, df_department, df_rootcause, df_overdue, df_effectiveness, df_trend


In [1]:
from pathlib import Path
import psycopg2
from sqlalchemy import create_engine
import pandas as pd

In [2]:
# Load Data
DATA_DIR = Path('data')
capa_df = pd.read_csv(DATA_DIR / 'capa_medfluss.csv')

In [3]:
capa_df.head()

,capa_id,company,recall_id,product_res_number,recalling_firm,product_description,nonconformity,original_root_cause,corrected_root_cause,department,assigned_owner,severity,capa_status,open_date,close_date,days_open,overdue,corrective_action,effectiveness_result
0,CAPA-MF-1000,MedFluss GmbH,85112,Z-0147-2010,Hospira Inc,Power cord for QVue Continuous Cardiac Output ...,Fire/Shock hazard-- The power cord used in the...,Component design/selection,Device Design,Design & Development,Mohammed Al-Farsi,Minor,Closed,2024-06-15,2024-09-02,79,No,Hospira initiated its recall on 08/14/2009. A...,Effective
1,CAPA-MF-1001,MedFluss GmbH,105064,Z-0268-2012,Wolf Medical Supply Inc.,"Wolf Medical Supply, Inc., WOLF-PAK REDI-FLO ...",Redi-Flo Elastomeric Infusion Pumps may have a...,Process control,Process control,Manufacturing,Carlos Rivera,Major,Open,2026-03-12,NaN,60,No,"On 10/20/2011 Wolf Medical Supply Inc., custom...",NaN
2,CAPA-MF-1002,MedFluss GmbH,105401,Z-0269-2012,Wolf Medical Supply Inc.,"Wolf Medical Supply Inc., WOLF-PAK REDI-FLO ...",Redi-Flo Elastomeric Infusion Pumps may have a...,Process control,Process control,Manufacturing,Lisa Hartmann,Major,Closed,2022-03-23,2022-06-01,70,No,"On 10/20/2011 Wolf Medical Supply Inc., custom...",Effective
3,CAPA-MF-1003,MedFluss GmbH,104072,Z-3284-2011,Hospira Inc.,Plum A+ Single Channel Infusion Pumps; Hospira...,Hospira has received reports of incorrect seat...,Nonconforming Material/Component,Nonconforming Material,Supplier Quality,Nina Scholz,Minor,Open,2025-07-28,NaN,287,Yes,"Hospira, Inc. sent an ""URGENT DEVICE RECALL"" l...",NaN
4,CAPA-MF-1004,MedFluss GmbH,107986,Z-1338-2012,Medtronic Neuromodulation,"Medtronic, Model 8870, Application Software Ca...",Medtronic has confirmed that an algorithm used...,Software design,Software Design,Design & Development,Emma Fischer,Major,Closed,2024-08-07,2024-10-16,70,No,"Medtronic mailed an ""Urgent Medical Device Cor...",Effective


In [4]:
# Database config
DB_NAME = 'medfluss'
DB_USER = 'jetalbhanarkar'
DB_HOST = 'localhost'
DB_PORT = '5432'

In [5]:
# Connect
conn = psycopg2.connect(
    dbname = DB_NAME,
    user = DB_USER,
    host = DB_HOST,
    port = DB_PORT
)

print('Connected to medfluss successfully')

Connected to medfluss successfully


In [6]:
from sqlalchemy import create_engine

In [7]:
engine = create_engine(f'postgresql://{DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

In [8]:
cursor = conn.cursor()
cursor.execute("DROP TABLE IF EXISTS capa_medfluss")
conn.commit()
cursor.close()
print("Table dropped")

Table dropped


In [9]:
# load CAPA table
capa_df.to_sql('capa_medfluss', engine, if_exists = 'replace', index = False)
print('CAPA table loaded successfully')

CAPA table loaded successfully


## KPI Queries

In [10]:
# CAPA Status Breakdown
query = """
SELECT capa_status,
    COUNT(*) AS  total_capas,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM capa_medfluss
GROUP BY capa_status
ORDER BY total_capas DESC;
"""

df_status = pd.read_sql(query, engine)
df_status

,capa_status,total_capas,percentage
0,Closed,421,79.58
1,Open,108,20.42


In [11]:
# Severity Breakdown
query = """
SELECT
    severity,
    COUNT(*) AS total_capas,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM capa_medfluss
GROUP BY severity
ORDER BY total_capas DESC;
"""

df_severity = pd.read_sql(query, engine)
df_severity

,severity,total_capas,percentage
0,Major,335,63.33
1,Minor,147,27.79
2,Critical,47,8.88


In [12]:
# Department Analysis
query = """
  SELECT
      department,
      COUNT(*) AS total_capas,
      SUM(CASE WHEN capa_status = 'Open' THEN 1 ELSE 0 END) AS open_capas,
      SUM(CASE WHEN capa_status = 'Closed' THEN 1 ELSE 0 END) AS closed_capas,
      SUM(CASE WHEN overdue = 'Yes' THEN 1 ELSE 0 END) AS overdue_capas
  FROM capa_medfluss
  GROUP BY department
  ORDER BY total_capas DESC;
  """

df_department = pd.read_sql(query, engine)
df_department

,department,total_capas,open_capas,closed_capas,overdue_capas
0,Design & Development,294,58,236,43
1,Manufacturing,125,27,98,21
2,Supplier Quality,93,18,75,13
3,Quality Assurance,9,2,7,2
4,Regulatory Affairs,8,3,5,3


In [13]:
# Root Cause Distribution
query = """
  SELECT
      corrected_root_cause,
      COUNT(*) AS total_capas,
      ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
  FROM capa_medfluss
  GROUP BY corrected_root_cause
  ORDER BY total_capas DESC
  LIMIT 10;
  """

df_rootcause = pd.read_sql(query, engine)
df_rootcause

,corrected_root_cause,total_capas,percentage
0,Device Design,217,41.02
1,Process control,98,18.53
2,Nonconforming Material,93,17.58
3,Software Design,77,14.56
4,Assembly Error,14,2.65
5,Human Error — Production,12,2.27
6,Release Without Testing,6,1.13
7,Labeling Design Error,4,0.76
8,Documentation Error,3,0.57
9,Under Investigation,3,0.57


# Overdue CAPAs by Severity and Department
query = """
SELECT
    severity,
    department,
    COUNT(*) AS total_open,
    SUM(CASE WHEN overdue = 'Yes' THEN 1 ELSE 0 END) AS overdue_count,
    ROUND(AVG(days_open)::numeric) AS avg_days_open
FROM capa_medfluss
WHERE capa_status = 'Open'
GROUP BY severity, department
ORDER BY overdue_count DESC;
"""

df_overdue = pd.read_sql(query, engine)
df_overdue

In [14]:
# Effectiveness of Closed CAPAs
query = """
SELECT
    effectiveness_result,
    COUNT(*) AS total,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM capa_medfluss
WHERE capa_status = 'Closed'
GROUP BY effectiveness_result
ORDER BY total DESC;
"""

df_effectiveness = pd.read_sql(query, engine)
df_effectiveness

,effectiveness_result,total,percentage
0,Effective,303,71.97
1,Partially Effective,79,18.76
2,Not Effective,39,9.26


In [15]:
# CAPA Trend by Year
query = """
SELECT
    TO_CHAR(open_date::date, 'YYYY') AS year,
    COUNT(*) AS capas_opened
FROM capa_medfluss
WHERE open_date IS NOT NULL
GROUP BY year
ORDER BY year;
"""

df_trend = pd.read_sql(query, engine)
df_trend

,year,capas_opened
0,2022,133
1,2023,131
2,2024,117
3,2025,109
4,2026,39
